# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, inspect, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata (do not subscript; use attribute access)
meta = dataset.metadata
print(f"{meta.name}\n\n{meta.description}\n")

## 2. Data Overview
Review available record sets and display key metadata for each, including their `@id` (the unique identifier required for further processing).

In [ ]:
# List all record sets (@id, name, fields) in the dataset
print("Available record sets in the dataset:")

record_set_summaries = []
for record_set in dataset.record_sets:
    print("- @id:", record_set.id)
    print("  name:", record_set.name)
    # List all fields in this record set by @id and name
    print("  fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id} | name: {field.name}")
    print()
    record_set_summaries.append({"id": record_set.id, "name": record_set.name, "fields": [(f.id, f.name) for f in record_set.fields]})

## 3. Data Extraction
Load records from the primary record set(s) into pandas DataFrames. All record sets and column/field references are by their `@id` as shown above.

In [ ]:
# Choose which record sets to extract (edit as desired; here we extract all)
record_set_ids = [r["id"] for r in record_set_summaries]

dataframes = {}

for record_set_id in record_set_ids:
    # Each record is a dict mapping field @id to value
    records = list(dataset.records(record_set=record_set_id))
    # Only load to DataFrame if not empty
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from record set @id: {record_set_id}")

# For demonstration, pick the first non-empty record set
if dataframes:
    first_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of columns in @{first_record_set_id}:\n")
    print(list(dataframes[first_record_set_id].columns))
    print("\nFirst five rows:")
    display(dataframes[first_record_set_id].head())
else:
    print("No records loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field and demonstrate filtering and normalization. All field/column references use the `@id` as listed in the overview.

In [ ]:
# Identify a numeric field (by inspection; modify as appropriate)
df = dataframes[first_record_set_id]
numeric_field_id = None

# Try to infer a numeric field by looking for integer/float columns
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    print(f"Selected numeric field: {numeric_field_id}")
    # Filter records
    threshold = df[numeric_field_id].quantile(0.75)  # use 75th percentile as example filter
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the values
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a likely categorical field
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found in first record set.")

## 5. Visualization
Visualize the distribution of the numeric field, and optionally visualize the normalized field or group statistics. (Make sure to run the previous EDA cell first.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True, color='teal')
    plt.title(f'Distribution of Numeric Field: {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If normalization has been done
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=12, kde=True, color='orange')
        plt.title(f'Normalized {numeric_field_id} (Filtered)')
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.show()
else:
    print("Visualization skipped: no numeric field available.")

## 6. Conclusion

- We demonstrated how to load and explore the FAIR² clinical oncology dataset using the Croissant format and the `mlcroissant` Python library.  
- All processing used record set and field references by their `@id`, ensuring precise and reproducible data access.  
- The workflow supports viewing metadata, extracting records to pandas DataFrames, filtering and analyzing numeric columns, and visualizing the data for inspection or downstream modeling.

For more advanced usage, see the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/python/reference.html) and adapt field/group/record references according to your analytical workflow!